In [4]:
from huggingface_hub import HfApi

api = HfApi()
for d in api.list_datasets(author="witfoo"):
    print(d.id)

witfoo/syslog-to-artifact
witfoo/witfoo-incidents
witfoo/witq-personality
witfoo/precinct6-cybersecurity
witfoo/precinct6-cybersecurity-100m


In [2]:
import sys
print(sys.executable)
import pandas, xgboost, shap, datasets
print("imports ok")

C:\Users\Aliya\microsoft_hackathon\venv\Scripts\python.exe
imports ok


In [6]:
from huggingface_hub import HfApi
from collections import defaultdict

api = HfApi()
for repo in ["witfoo/precinct6-cybersecurity", "witfoo/precinct6-cybersecurity-100m"]:
    print("=====", repo)
    info = api.dataset_info(repo, files_metadata=True)

    # 1) count files and total size per top-level folder
    folders = defaultdict(lambda: [0, 0.0])
    for f in info.siblings:
        top = f.rfilename.split("/")[0] if "/" in f.rfilename else "(root)"
        folders[top][0] += 1
        folders[top][1] += (f.size or 0) / 1e6
    print("-- folders: name | files | total MB")
    for name, (n, mb) in sorted(folders.items()):
        print(f"{name} | {n} | {mb:.1f}")

    # 2) list only files bigger than 1 MB
    print("-- files bigger than 1 MB")
    for f in info.siblings:
        if f.size and f.size > 1e6:
            print(f"{f.rfilename} | {f.size/1e6:.1f} MB")

===== witfoo/precinct6-cybersecurity
-- folders: name | files | total MB
(root) | 2 | 0.0
build | 4 | 2.9
graph | 51380 | 8283.7
reference | 1 | 0.2
signals | 3 | 127.6
-- files bigger than 1 MB
build/label_stats.json | 2.7 MB
graph/attack_reports.jsonl | 92.5 MB
graph/attack_reports.parquet | 4.2 MB
graph/edges-00000.parquet | 35.8 MB
graph/edges.jsonl | 1708.6 MB
graph/graph.graphml | 1396.2 MB
graph/incidents.jsonl | 4490.7 MB
graph/nodes.jsonl | 32.4 MB
graph/nodes.parquet | 4.3 MB
signals/incident_signals.parquet | 10.5 MB
signals/signals.parquet | 117.1 MB
===== witfoo/precinct6-cybersecurity-100m
-- folders: name | files | total MB
(root) | 2 | 0.0
build | 3 | 2.8
graph | 51394 | 9006.6
reference | 1 | 0.2
signals | 60 | 4365.6
-- files bigger than 1 MB
build/label_stats.json | 2.8 MB
graph/attack_reports.jsonl | 92.5 MB
graph/attack_reports.parquet | 4.2 MB
graph/edges-00000.jsonl.gz | 254.7 MB
graph/edges-00000.parquet | 260.3 MB
graph/edges-00001.jsonl.gz | 241.4 MB
graph/edg

In [7]:
from pathlib import Path
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

# find the project folder, whether the notebook runs from notebooks/ or the root
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_dir = root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

path = hf_hub_download(
    repo_id="witfoo/precinct6-cybersecurity",
    repo_type="dataset",
    filename="signals/signals.parquet",
    local_dir=raw_dir,
)
print("saved to:", path)

# read only the file's header (no data loaded)
pf = pq.ParquetFile(path)
print("rows:", pf.metadata.num_rows)
print("columns:", len(pf.schema_arrow.names))
for field in pf.schema_arrow:
    print(field.name, "|", field.type)

signals/signals.parquet: reconstructing file:   0%|          |  0.00B /  117MB            

signals/signals.parquet: downloading bytes:           |  0.00B            

saved to: C:\Users\Aliya\microsoft_hackathon\data\raw\signals\signals.parquet
rows: 2011674
columns: 38
timestamp | double
event_time | double
timestamp_source | string
origin | string
org_id | string
artifact_id | string
message_type | string
stream_name | string
pipeline | string
src_ip | string
dst_ip | string
src_port | string
dst_port | string
protocol | string
src_host | string
dst_host | string
username | string
action | string
severity | string
vendor_code | string
message_sanitized | string
label_binary | string
label_confidence | double
attack_techniques | string
attack_tactics | string
defense_techniques | string
mo_name | string
suspicion_score | double
lifecycle_stage | string
disposition | string
disposition_category | string
is_false_positive | bool
status_name | string
incident_ids | string
matched_rules | string
set_roles | string
product_name | string
vendor_name | string


In [8]:
from pathlib import Path
from collections import defaultdict
from huggingface_hub import HfApi, hf_hub_download

REPO = "witfoo/precinct6-cybersecurity"
RAW = Path(r"C:\Users\Aliya\microsoft_hackathon\data\raw")

# 1. Summary by top-level folder
info = HfApi().dataset_info(REPO, files_metadata=True)
by_folder = defaultdict(lambda: [0, 0.0])
for f in info.siblings:
    folder = f.rfilename.split("/")[0] if "/" in f.rfilename else "(root)"
    by_folder[folder][0] += 1
    by_folder[folder][1] += (f.size or 0) / 1e6
for k, (n, mb) in by_folder.items():
    print(f"{k:20s} files={n:4d}  total={mb:9.1f} MB")

# 2. Download ONE signals shard (about 70 MB)
first = sorted(f.rfilename for f in info.siblings if f.rfilename.startswith("signals/"))[0]
print("downloading:", first)
path = hf_hub_download(REPO, first, repo_type="dataset", local_dir=RAW)
print("saved to:", path)

Task was destroyed but it is pending!
task: <Task pending name='Task-136' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\Aliya\microsoft_hackathon\venv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-137' coro=<Kernel.shell_main() running at C:\Users\Aliya\microsoft_hackathon\venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\Aliya\microsoft_hackathon\venv\Lib\site-packages\zmq\eventloop\zmqstream.py:564]>
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3824.0_x64__qbz5n2kfra8p0\Lib\json\decoder.py:361: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  obj, end = self.scan_once(s, idx)
Task was destroyed but it is pending!
task: <Task pending name='Task-137' coro=<Kernel.shell_main() running at C:\Users\Aliya\microsoft_hackathon\venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]>


(root)               files=   2  total=      0.0 MB
build                files=   4  total=      2.9 MB
graph                files=51380  total=   8283.7 MB
reference            files=   1  total=      0.2 MB
signals              files=   3  total=    127.6 MB
downloading: signals/incident_signals.parquet


signals/incident_signals.parquet: reconstructing file:   0%|          |  0.00B / 10.5MB            

signals/incident_signals.parquet: downloading bytes:           |  0.00B            

saved to: C:\Users\Aliya\microsoft_hackathon\data\raw\signals\incident_signals.parquet


In [9]:
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

REPO = "witfoo/precinct6-cybersecurity"
RAW = Path(r"C:\Users\Aliya\microsoft_hackathon\data\raw")

info = HfApi().dataset_info(REPO, files_metadata=True)
for f in sorted(info.siblings, key=lambda x: x.rfilename):
    if not f.rfilename.startswith("graph/"):      # skip the 51k graph files
        print(f"{f.rfilename:60s} {(f.size or 0)/1e6:8.1f} MB")

# Read the dataset card (README) so we use the documented column meanings
readme = hf_hub_download(REPO, "README.md", repo_type="dataset", local_dir=RAW)
text = Path(readme).read_text(encoding="utf-8")
print("\n--- README length:", len(text), "chars ---")
print(text[:6000])

.gitattributes                                                    0.0 MB
README.md                                                         0.0 MB
build/label_stats.json                                            2.7 MB
build/sanitize_stats.json                                         0.0 MB
build/subset_stats.json                                           0.1 MB
build/verification_report.json                                    0.0 MB
reference/lead_rules_catalog.json                                 0.2 MB
signals/incident_signals.parquet                                 10.5 MB
signals/metadata.json                                             0.0 MB
signals/signals.parquet                                         117.1 MB


README.md:   0%|          | 0.00/26.1k [00:00<?, ?B/s]


--- README length: 25986 chars ---
---
license: apache-2.0
task_categories:
  - text-classification
  - graph-ml
language:
  - en
tags:
  - cybersecurity
  - intrusion-detection
  - provenance-graphs
  - MITRE-ATT&CK
  - SOAR
  - security-operations
  - IDS
  - network-security
  - threat-detection
  - labeled-dataset
  - lead-rules
pretty_name: WitFoo Precinct6 Cybersecurity Dataset
size_categories:
  - 1M<n<10M
dataset_info:
  - config_name: signals
    splits:
      - name: train
        num_examples: 2011674
  - config_name: incident_signals
    splits:
      - name: train
        num_examples: 238511
configs:
  - config_name: signals
    data_files:
      - split: train
        path: signals/signals.parquet
  - config_name: incident_signals
    data_files:
      - split: train
        path: signals/incident_signals.parquet
  - config_name: graph_nodes
    data_files:
      - split: train
        path: graph/nodes.parquet
  - config_name: graph_edges
    data_files:
      - split:

In [10]:
import re
from pathlib import Path
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

REPO = "witfoo/precinct6-cybersecurity"
RAW = Path(r"C:\Users\Aliya\microsoft_hackathon\data\raw")

# 1. Download the main table (~117 MB)
p = hf_hub_download(REPO, "signals/signals.parquet", repo_type="dataset", local_dir=RAW)
print("saved to:", p)

# 2. Schema only (no data loaded)
pf = pq.ParquetFile(p)
print("rows:", pf.metadata.num_rows, "| columns:", pf.metadata.num_columns, "| row groups:", pf.metadata.num_row_groups)
for field in pf.schema_arrow:
    print(f"{field.name:40s} {field.type}")

# 3. README headings only
readme = (RAW / "README.md").read_text(encoding="utf-8")
print("\n--- README headings ---")
for line in readme.splitlines():
    if re.match(r"^#{1,4} ", line):
        print(line)

saved to: C:\Users\Aliya\microsoft_hackathon\data\raw\signals\signals.parquet
rows: 2011674 | columns: 38 | row groups: 41
timestamp                                double
event_time                               double
timestamp_source                         string
origin                                   string
org_id                                   string
artifact_id                              string
message_type                             string
stream_name                              string
pipeline                                 string
src_ip                                   string
dst_ip                                   string
src_port                                 string
dst_port                                 string
protocol                                 string
src_host                                 string
dst_host                                 string
username                                 string
action                                   string
severity     

In [11]:
import re
from pathlib import Path
import pandas as pd

RAW = Path(r"C:\Users\Aliya\microsoft_hackathon\data\raw")
text = (RAW / "README.md").read_text(encoding="utf-8")

def section(title, maxchars=3500):
    m = re.search(rf"^(#{{2,3}}) {re.escape(title)}\s*$", text, re.M)
    if not m:
        print(f"[not found: {title}]"); return
    level = len(m.group(1))
    rest = text[m.end():]
    n = re.search(rf"^#{{1,{level}}} ", rest, re.M)
    body = rest[:n.start()] if n else rest
    print(f"\n===== {title} =====\n{body[:maxchars]}")

for t in ["Label distribution", "Signal columns", "Labeling methodology", "Ground truth and disposition"]:
    section(t)

# Label counts (loads only 4 columns)
cols = ["label_binary", "disposition_category", "is_false_positive", "severity"]
df = pd.read_parquet(RAW / "signals" / "signals.parquet", columns=cols)
for c in cols:
    print(f"\n--- {c} ---")
    print(df[c].value_counts(dropna=False).head(10))


===== Label distribution =====

Across both tables (2,250,185 rows):

| Label | Rows | Share |
|---|---|---|
| `benign` | 1,897,153 | 84.31% |
| `suspicious` | 106,793 | 4.75% |
| `malicious` | 246,239 | 10.94% |

Disposition of malicious rows (raw Precinct incident status, see [Ground Truth](#ground-truth-and-disposition)):

`signals`:

| Disposition | Rows |
|---|---|
| `Cold Case` | 3 |
| `Disrupted` | 5,779 |
| `Open` | 15 |
| `Unprocessed` | 1,931 |

`incident_signals`:

| Disposition | Rows |
|---|---|
| `Cold Case` | 14 |
| `Dismissed` | 5 |
| `Disrupted` | 176,510 |
| `Open` | 330 |
| `Unprocessed` | 61,652 |



===== Signal columns =====

Both signal tables share one schema (38 columns).

| Column | Type | Description |
|--------|------|-------------|
| `timestamp` | float | Unix epoch seconds. Live rows: Precinct ingest time (artifact timeuuid), seconds after the event. Incident leads: when Precinct's correlation raised the lead, hours to days after the event, at one-second 

In [12]:
import pandas as pd
from pathlib import Path

RAW = Path(r"C:\Users\Aliya\microsoft_hackathon\data\raw")
cols = ["timestamp", "event_time", "org_id", "message_type", "label_binary",
        "attack_techniques", "defense_techniques", "mo_name"]
df = pd.read_parquet(RAW / "signals" / "signals.parquet", columns=cols)
df["ts"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

print("time span:", df.ts.min(), "->", df.ts.max())
print("\nrows per week x label:")
print(df.groupby([df.ts.dt.to_period("W"), "label_binary"]).size().unstack(fill_value=0))

print("\nrows per org x label:")
print(df.groupby(["org_id", "label_binary"]).size().unstack(fill_value=0))

def nonempty(s):
    return s.notna() & ~s.isin(["", "[]", "null"])

print("\nshare non-empty, by label:")
print(pd.DataFrame({
    "attack_techniques": nonempty(df.attack_techniques),
    "defense_techniques": nonempty(df.defense_techniques),
    "mo_name": nonempty(df.mo_name),
    "event_time": df.event_time.notna(),
}).groupby(df.label_binary).mean().round(3))

print("\ntop message_type among malicious:")
print(df.loc[df.label_binary == "malicious", "message_type"].value_counts().head(8))

time span: 2024-07-26 11:10:23.729254961+00:00 -> 2024-08-01 06:00:03.239708900+00:00

rows per week x label:


C:\Users\Aliya\AppData\Local\Temp\ipykernel_35188\487361388.py:12: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  print(df.groupby([df.ts.dt.to_period("W"), "label_binary"]).size().unstack(fill_value=0))


label_binary            benign  malicious  suspicious
ts                                                   
2024-07-22/2024-07-28  1227276       4176       95031
2024-07-29/2024-08-04   669877       3552       11762

rows per org x label:
label_binary  benign  malicious  suspicious
org_id                                     
ORG-0001        4958          0          10
ORG-0003      433775         16         113
ORG-0004      976694       4176       15115
ORG-0005      481726       3536       91555

share non-empty, by label:
              attack_techniques  defense_techniques  mo_name  event_time
label_binary                                                            
benign                      0.0                 0.0      0.0       0.260
malicious                   1.0                 0.0      1.0       0.001
suspicious                  1.0                 0.0      0.0       0.000

top message_type among malicious:
message_type
network_flow_data            4156
firewall_action       

In [13]:
import re
import pandas as pd
from pathlib import Path

RAW = Path(r"C:\Users\Aliya\microsoft_hackathon\data\raw")
df = pd.read_parquet(RAW / "signals" / "signals.parquet",
                     columns=["timestamp", "org_id", "message_type", "label_binary",
                              "attack_techniques", "mo_name"])
df["day"] = pd.to_datetime(df["timestamp"], unit="s").dt.strftime("%Y-%m-%d")

print("=== rows per day x org x label ===")
print(df.groupby(["day", "org_id", "label_binary"]).size().unstack(fill_value=0))

print("\n=== message_type: is it a giveaway? (top by malicious count) ===")
mt = df.groupby("message_type")["label_binary"].value_counts().unstack(fill_value=0)
mt["mal_share"] = (mt["malicious"] / mt[["benign", "malicious", "suspicious"]].sum(axis=1)).round(4)
print(mt.sort_values("malicious", ascending=False).head(8))

mal = df[df.label_binary == "malicious"]
print("\n=== malicious: attack_techniques (top 10) ===")
print(mal.attack_techniques.value_counts().head(10))
print("\n=== malicious: mo_name (top 8) ===")
print(mal.mo_name.value_counts().head(8))

print("\n=== incident_signals: is defense_techniques filled there? ===")
inc = pd.read_parquet(RAW / "signals" / "incident_signals.parquet",
                      columns=["label_binary", "defense_techniques", "attack_techniques"])
ne = lambda s: s.notna() & ~s.isin(["", "[]", "null"])
print(inc.groupby("label_binary")[["defense_techniques", "attack_techniques"]].agg(lambda s: ne(s).mean()).round(3))
print(inc.loc[ne(inc.defense_techniques), "defense_techniques"].head(3).tolist())

# README sections on attack reports and files
text = (RAW / "README.md").read_text(encoding="utf-8")
def section(title, maxchars=2500):
    m = re.search(rf"^(#{{2,3}}) {re.escape(title)}\s*$", text, re.M)
    if not m:
        print(f"[not found: {title}]"); return
    rest = text[m.end():]
    n = re.search(rf"^#{{1,{len(m.group(1))}}} ", rest, re.M)
    print(f"\n===== {title} =====\n{(rest[:n.start()] if n else rest)[:maxchars]}")
for t in ["Attack reports", "Files"]:
    section(t)

=== rows per day x org x label ===
label_binary         benign  malicious  suspicious
day        org_id                                 
2024-07-26 ORG-0001    2690          0           7
           ORG-0003  138934          3          59
           ORG-0004  159268        603        2393
           ORG-0005  259642       2654       47660
2024-07-27 ORG-0001    2268          0           3
           ORG-0003  294841         13          54
           ORG-0004  136727         21         960
           ORG-0005  222084        882       43895
2024-07-28 ORG-0004   10822          0           0
2024-07-29 ORG-0004  397315       3515       10734
2024-07-30 ORG-0004  161715         37         816
2024-07-31 ORG-0004   87364          0         188
2024-08-01 ORG-0004   23483          0          24

=== message_type: is it a giveaway? (top by malicious count) ===
label_binary               benign  malicious  suspicious  mal_share
message_type                                                      

In [14]:
import pandas as pd
from pathlib import Path

RAW = Path(r"C:\Users\Aliya\microsoft_hackathon\data\raw")
feat_cols = ["message_type", "stream_name", "pipeline", "product_name", "vendor_name",
             "action", "protocol", "severity", "vendor_code"]
cols = ["label_binary", "attack_techniques", "incident_ids", "src_ip", "dst_ip", "dst_port"] + feat_cols
df = pd.read_parquet(RAW / "signals" / "signals.parquet", columns=cols)
df["nonbenign"] = df.label_binary != "benign"

def one_sided(d, columns, pos_mask):
    """Share of positive/negative rows sitting in values that never occur in the other class."""
    out = []
    for c in columns:
        t = pd.crosstab(d[c].fillna("NA"), pos_mask).reindex(columns=[False, True], fill_value=0)
        out.append({
            "column": c, "n_unique": len(t),
            "pos_rows_in_pos_only_values": round(t.loc[t[False] == 0, True].sum() / max(t[True].sum(), 1), 3),
            "neg_rows_in_neg_only_values": round(t.loc[t[True] == 0, False].sum() / max(t[False].sum(), 1), 3),
        })
    return pd.DataFrame(out)

print("=== A) non-benign (pos) vs benign (neg) ===")
print(one_sided(df, feat_cols, df["nonbenign"]))

nb = df[df.nonbenign]
print("\n=== B) malicious (pos) vs suspicious (neg), within non-benign ===")
print(one_sided(nb, feat_cols, nb.label_binary == "malicious"))

print("\n=== benign: top message_type ===")
print(df.loc[~df.nonbenign, "message_type"].value_counts().head(8))

mal = df[df.label_binary == "malicious"]
print("\n=== malicious: distinct incident_ids strings:", mal.incident_ids.nunique())
print(mal.incident_ids.value_counts().head(5))
print("distinct src_ip / dst_ip / dst_port:", mal.src_ip.nunique(), mal.dst_ip.nunique(), mal.dst_port.nunique())

sus = df[df.label_binary == "suspicious"]
print("\n=== suspicious: distinct attack_techniques:", sus.attack_techniques.nunique())
print(sus.attack_techniques.value_counts().head(8))

text = (RAW / "README.md").read_text(encoding="utf-8")
for kw in ["D3FEND", "remediat", "mitigation"]:
    hits = [l.strip()[:160] for l in text.splitlines() if kw.lower() in l.lower()]
    print(f"\n[{kw}] {len(hits)} lines")
    print("\n".join(hits[:5]))

=== A) non-benign (pos) vs benign (neg) ===
         column  n_unique  pos_rows_in_pos_only_values  \
0  message_type       595                        1.000   
1   stream_name        82                        0.168   
2      pipeline         3                        0.000   
3  product_name        20                        0.168   
4   vendor_name        13                        0.000   
5        action        44                        0.998   
6      protocol         6                        0.000   
7      severity        10                        0.000   
8   vendor_code    463831                        0.831   

   neg_rows_in_neg_only_values  
0                        1.000  
1                        0.770  
2                        0.287  
3                        0.770  
4                        0.305  
5                        0.221  
6                        0.000  
7                        0.010  
8                        0.541  

=== B) malicious (pos) vs suspicious (neg), 

In [15]:
import json
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

REPO = "witfoo/precinct6-cybersecurity"
RAW = Path(r"C:\Users\Aliya\microsoft_hackathon\data\raw")

def peek(name, x, n=1800):
    print(f"\n===== {name}: {type(x).__name__}, len={len(x) if hasattr(x, '__len__') else '-'} =====")
    if isinstance(x, dict):
        for k, v in list(x.items())[:12]:
            print(f"  {k}: {type(v).__name__}" + (f" (len {len(v)})" if hasattr(v, '__len__') else ""))
        first = next(iter(x.values()))
    else:
        first = x[0] if x else None
    print("first item ->", json.dumps(first, indent=1, default=str)[:n])

for rel in ["reference/lead_rules_catalog.json", "signals/metadata.json"]:
    p = hf_hub_download(REPO, rel, repo_type="dataset", local_dir=RAW)
    peek(rel, json.loads(Path(p).read_text(encoding="utf-8")))

print("\n===== graph/ files other than per-incident graphml =====")
info = HfApi().dataset_info(REPO, files_metadata=True)
for f in sorted(info.siblings, key=lambda s: s.rfilename):
    if f.rfilename.startswith("graph/") and "incidents_graphml" not in f.rfilename:
        print(f"{f.rfilename:45s} {(f.size or 0)/1e6:9.1f} MB")

lead_rules_catalog.json:   0%|          | 0.00/150k [00:00<?, ?B/s]


===== reference/lead_rules_catalog.json: dict, len=4 =====
  products: dict (len 158)
  stream_to_product: dict (len 216)
  lead_rules: list (len 261)
  sets: dict (len 106)
first item -> {
 "1": {
  "name": "Stealthwatch",
  "vendor": "Cisco",
  "streams": [
   "Stealthwatch High Concern Index",
   "Stealthwatch Suspect Data Hoarding Alarm",
   "Stealthwatch Suspect Data Hoarding",
   "Stealthwatch Suspect Data Loss",
   "Stealthwatch Suspect Quiet Long Flow",
   "stealth_watch",
   "lancope_stealthwatch"
  ],
  "frameworks": [
   "csc",
   "cmmc1",
   "cmmc3",
   "pci32",
   "nist80053",
   "csf",
   "nist800171",
   "csc8",
   "pci40",
   "iso27001",
   "soc2"
  ]
 },
 "2": {
  "name": "QRadar",
  "vendor": "IBM",
  "streams": [
   "QRadar Offense"
  ],
  "frameworks": [
   "csc",
   "cmmc1",
   "cmmc3",
   "pci32",
   "nist80053",
   "csf",
   "nist800171",
   "csc8",
   "pci40",
   "iso27001",
   "soc2"
  ]
 },
 "3": {
  "name": "Advanced Malware Protection (AMP)",
  "vendor": "C

metadata.json:   0%|          | 0.00/10.1k [00:00<?, ?B/s]


===== signals/metadata.json: dict, len=6 =====
  schema_version: str (len 3)
  columns: list (len 38)
  total_records: int
  label_distribution: dict (len 3)
  tables: dict (len 2)
  formats: list (len 1)
first item -> "2.0"

===== graph/ files other than per-incident graphml =====
graph/attack_reports.jsonl                         92.5 MB
graph/attack_reports.parquet                        4.2 MB
graph/edges-00000.parquet                          35.8 MB
graph/edges.jsonl                                1708.6 MB
graph/graph.graphml                              1396.2 MB
graph/incidents.jsonl                            4490.7 MB
graph/metadata.json                                 0.0 MB
graph/nodes.jsonl                                  32.4 MB
graph/nodes.parquet                                 4.3 MB


In [16]:
import json
from pathlib import Path

RAW = Path(r"C:\Users\Aliya\microsoft_hackathon\data\raw")
cat = json.loads((RAW / "reference" / "lead_rules_catalog.json").read_text(encoding="utf-8"))
meta = json.loads((RAW / "signals" / "metadata.json").read_text(encoding="utf-8"))

rules = cat["lead_rules"]
print("=== lead_rules: keys of first rule ===")
print(list(rules[0].keys()) if isinstance(rules[0], dict) else type(rules[0]))
print("\n=== first 3 rules ===")
print(json.dumps(rules[:3], indent=1, default=str)[:2500])

print("\n=== sets: first 3 ===")
items = list(cat["sets"].items())[:3]
print(json.dumps(items, indent=1, default=str)[:1200])

print("\n=== signals/metadata.json ===")
for k in ["label_distribution", "tables"]:
    print(f"\n-- {k} --")
    print(json.dumps(meta[k], indent=1, default=str)[:2500])

=== lead_rules: keys of first rule ===
['id', 'description', 'criteria', 'client_set_id', 'server_set_id', 'product_id']

=== first 3 rules ===
[
 {
  "id": 1,
  "description": "Stealthwatch High Concern Index Alarm",
  "criteria": "streamName = \"Stealthwatch High Concern Index\"",
  "client_set_id": 1001,
  "server_set_id": 1005,
  "product_id": 1
 },
 {
  "id": 2,
  "description": "Stealthwatch Suspect Data Hoarding Alarm",
  "criteria": "streamName = \"Stealthwatch Suspect Data Hoarding\"",
  "client_set_id": 1002,
  "server_set_id": 1006,
  "product_id": 1
 },
 {
  "id": 3,
  "description": "Stealthwatch Suspect Data Loss Alarm",
  "criteria": "streamName = \"Stealthwatch Suspect Data Loss\"",
  "client_set_id": 1003,
  "server_set_id": 1007,
  "product_id": 1
 }
]

=== sets: first 3 ===
[
 [
  "1",
  "Exploiting Host"
 ],
 [
  "2",
  "Staging Host"
 ],
 [
  "3",
  "Exfiltration Host"
 ]
]

=== signals/metadata.json ===

-- label_distribution --
{
 "benign": 1897153,
 "malicious":

In [17]:
import json
from pathlib import Path

RAW = Path(r"C:\Users\Aliya\microsoft_hackathon\data\raw")
cat = json.loads((RAW / "reference" / "lead_rules_catalog.json").read_text(encoding="utf-8"))
meta = json.loads((RAW / "signals" / "metadata.json").read_text(encoding="utf-8"))

rules = cat["lead_rules"]
print("=== lead_rules: keys of first rule ===")
print(list(rules[0].keys()) if isinstance(rules[0], dict) else type(rules[0]))
print("\n=== first 3 rules ===")
print(json.dumps(rules[:3], indent=1, default=str)[:2500])

print("\n=== sets: first 3 ===")
items = list(cat["sets"].items())[:3]
print(json.dumps(items, indent=1, default=str)[:1200])

print("\n=== signals/metadata.json ===")
for k in ["label_distribution", "tables"]:
    print(f"\n-- {k} --")
    print(json.dumps(meta[k], indent=1, default=str)[:2500])

=== lead_rules: keys of first rule ===
['id', 'description', 'criteria', 'client_set_id', 'server_set_id', 'product_id']

=== first 3 rules ===
[
 {
  "id": 1,
  "description": "Stealthwatch High Concern Index Alarm",
  "criteria": "streamName = \"Stealthwatch High Concern Index\"",
  "client_set_id": 1001,
  "server_set_id": 1005,
  "product_id": 1
 },
 {
  "id": 2,
  "description": "Stealthwatch Suspect Data Hoarding Alarm",
  "criteria": "streamName = \"Stealthwatch Suspect Data Hoarding\"",
  "client_set_id": 1002,
  "server_set_id": 1006,
  "product_id": 1
 },
 {
  "id": 3,
  "description": "Stealthwatch Suspect Data Loss Alarm",
  "criteria": "streamName = \"Stealthwatch Suspect Data Loss\"",
  "client_set_id": 1003,
  "server_set_id": 1007,
  "product_id": 1
 }
]

=== sets: first 3 ===
[
 [
  "1",
  "Exploiting Host"
 ],
 [
  "2",
  "Staging Host"
 ],
 [
  "3",
  "Exfiltration Host"
 ]
]

=== signals/metadata.json ===

-- label_distribution --
{
 "benign": 1897153,
 "malicious":

In [19]:
print("##### POOL INSPECTION #####")
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

RAW = Path(r"C:\Users\Aliya\microsoft_hackathon\data\raw")
cols = ["timestamp", "org_id", "label_binary", "message_type", "stream_name", "product_name",
        "action", "protocol", "src_ip", "dst_ip", "src_port", "dst_port", "src_host", "dst_host",
        "username", "severity", "vendor_code", "message_sanitized"]
df = pd.read_parquet(RAW / "signals" / "signals.parquet", columns=cols)
pool = df[df.label_binary != "benign"].copy()
pool["day"] = pd.to_datetime(pool["timestamp"], unit="s").dt.strftime("%m-%d")
print("pool shape:", pool.shape, "| malicious share:", round((pool.label_binary == "malicious").mean(), 4))

show = ["day", "org_id", "message_type", "stream_name", "action", "protocol",
        "src_ip", "dst_ip", "dst_port", "severity"]
for lab in ["malicious", "suspicious"]:
    s = pool[pool.label_binary == lab].sample(4, random_state=42)
    print(f"\n=== {lab}: 4 random rows ===")
    print(s[show].to_string())
    for m in s.message_sanitized:
        print("  MSG:", str(m)[:220])

print("\n=== top streams in pool (rows by label) ===")
print(pool.groupby(["stream_name", "label_binary"]).size().unstack(fill_value=0)
          .sort_values("malicious", ascending=False).head(10))

print("\n=== share empty (null or blank) by column, by label ===")
empty = pool[cols].apply(lambda s: s.isna() | (s.astype(str).str.strip() == ""))
print(empty.groupby(pool.label_binary).mean().T.round(3))

##### POOL INSPECTION #####


ArrowMemoryError: realloc of size 138412032 failed

In [1]:
print("##### POOL INSPECTION v2 #####")
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

ROOT = Path(r"C:\Users\Aliya\microsoft_hackathon")
RAW, INTERIM = ROOT / "data" / "raw", ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

cols = ["timestamp", "org_id", "label_binary", "message_type", "stream_name", "product_name",
        "action", "protocol", "src_ip", "dst_ip", "src_port", "dst_port", "src_host", "dst_host",
        "username", "severity", "vendor_code", "message_sanitized"]

# Read one row group at a time and keep only non-benign rows
pf = pq.ParquetFile(RAW / "signals" / "signals.parquet")
parts = []
for i in range(pf.metadata.num_row_groups):
    g = pf.read_row_group(i, columns=cols).to_pandas()
    parts.append(g[g.label_binary != "benign"])
    del g
pool = pd.concat(parts, ignore_index=True)
del parts

pool["day"] = pd.to_datetime(pool["timestamp"], unit="s").dt.strftime("%m-%d")
pool.to_parquet(INTERIM / "pool_raw.parquet", index=False)   # saved so we never reload the big file
print("pool shape:", pool.shape, "| malicious share:", round((pool.label_binary == "malicious").mean(), 4))

show = ["day", "org_id", "message_type", "stream_name", "action", "protocol",
        "src_ip", "dst_ip", "dst_port", "severity"]
for lab in ["malicious", "suspicious"]:
    s = pool[pool.label_binary == lab].sample(4, random_state=42)
    print(f"\n=== {lab}: 4 random rows ===")
    print(s[show].to_string())
    for m in s.message_sanitized:
        print("  MSG:", str(m)[:220])

print("\n=== top streams in pool (rows by label) ===")
print(pool.groupby(["stream_name", "label_binary"]).size().unstack(fill_value=0)
          .sort_values("malicious", ascending=False).head(10))

print("\n=== share empty (null or blank) by column, by label ===")
empty = pool[cols].apply(lambda s: s.isna() | (s.astype(str).str.strip() == ""))
print(empty.groupby(pool.label_binary).mean().T.round(3))

##### POOL INSPECTION v2 #####
pool shape: (114521, 19) | malicious share: 0.0675

=== malicious: 4 random rows ===
         day    org_id       message_type       stream_name action protocol         src_ip          dst_ip dst_port severity
27228  07-27  ORG-0005    firewall_action         cisco_asa  block        6  100.64.29.205  10.147.100.191     9926  warning
53149  07-26  ORG-0005    firewall_action         cisco_asa  block       17  100.64.30.130  10.147.100.191     1027  warning
76299  07-26  ORG-0005    firewall_action         cisco_asa  block       17  10.47.200.140   198.51.100.31       53  warning
597    07-29  ORG-0004  network_flow_data  aws_vpc_flow_log  block        6  100.64.27.187   10.98.236.105    10443         
  MSG: <164>Jul 26 2024 22:44:25: %ASA-4-106023: Deny tcp src outside:100.64.29.205/51023 dst dmz-1:10.147.100.191/9926 by access-group "outside_access_in" [0x0, 0x0]

  MSG: <164>Jul 26 2024 06:41:15: %ASA-4-106023: Deny udp src outside:100.64.30.130/54706 d

In [2]:
print("##### POOL CHECK 2 #####")
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

ROOT = Path(r"C:\Users\Aliya\microsoft_hackathon")
pool = pd.read_parquet(ROOT / "data" / "interim" / "pool_raw.parquet")
pool["y"] = (pool.label_binary == "malicious").astype(int)
pool["stream_grp"] = np.where(pool.stream_name.isin(["cisco_asa", "aws_vpc_flow_log"]),
                              pool.stream_name, "other")

print("A) day x org x stream: malicious / rows")
t = pool.groupby(["day", "org_id", "stream_grp"]).y.agg(malicious="sum", rows="count")
print(t)

print("\nB) cisco_asa message IDs by label")
asa = pool[pool.stream_name == "cisco_asa"].copy()
asa["asa_id"] = asa.message_sanitized.str.extract(r"%ASA-\d-(\d+)")[0].fillna("no_match")
print(asa.groupby(["asa_id", "label_binary"]).size().unstack(fill_value=0))

print("\nC) aws_vpc_flow_log: token layout check")
vpc = pool[pool.stream_name == "aws_vpc_flow_log"].copy()
parts = vpc.message_sanitized.str.split(" ", expand=True)
print("tokens per message:", parts.notna().sum(axis=1).value_counts().head(3).to_dict())
print(parts.head(3).to_string())
vpc["packets"] = pd.to_numeric(parts[8], errors="coerce")
vpc["bytes"] = pd.to_numeric(parts[9], errors="coerce")
print(vpc.groupby("label_binary")[["packets", "bytes"]].agg(["median", "mean", "max"]))
print("action token:", parts[12].value_counts().head(3).to_dict())

print("\nD) source-IP overlap")
mal = pool[pool.y == 1]
sus_ips = set(pool.loc[pool.y == 0, "src_ip"])
print("distinct malicious src_ip:", mal.src_ip.nunique(),
      "| malicious rows whose src_ip also appears in suspicious rows:",
      round(mal.src_ip.isin(sus_ips).mean(), 3))
train_ips = set(pool.loc[pool.day == "07-26", "src_ip"])
test_mal = pool[pool.day.isin(["07-29", "07-30", "07-31", "08-01"]) & (pool.y == 1)]
print("test-period malicious rows whose src_ip was seen on 07-26:",
      round(test_mal.src_ip.isin(train_ips).mean(), 3), "of", len(test_mal))

##### POOL CHECK 2 #####
A) day x org x stream: malicious / rows
                                 malicious   rows
day   org_id   stream_grp                        
07-26 ORG-0001 other                     0      7
      ORG-0003 other                     3     62
      ORG-0004 aws_vpc_flow_log        593   2978
               other                    10     18
      ORG-0005 cisco_asa              2651  50301
               other                     3     13
07-27 ORG-0001 other                     0      3
      ORG-0003 other                    13     67
      ORG-0004 aws_vpc_flow_log         18    970
               other                     3     11
      ORG-0005 cisco_asa               874  44761
               other                     8     16
07-29 ORG-0004 aws_vpc_flow_log       3508  14218
               other                     7     31
07-30 ORG-0004 aws_vpc_flow_log         37    845
               other                     0      8
07-31 ORG-0004 aws_vpc_flow_log    

In [3]:
print("##### POOL CHECK 3 #####")
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

ROOT = Path(r"C:\Users\Aliya\microsoft_hackathon")
SIG = ROOT / "data" / "raw" / "signals" / "signals.parquet"
INTERIM = ROOT / "data" / "interim"
pool = pd.read_parquet(INTERIM / "pool_raw.parquet")

# 1) Re-read in the same order to attach incident_ids; collect benign IPs on the way
pf = pq.ParquetFile(SIG)
meta_parts, ben_src, ben_dst = [], set(), set()
for i in range(pf.metadata.num_row_groups):
    g = pf.read_row_group(i, columns=["label_binary", "src_ip", "dst_ip", "incident_ids"]).to_pandas()
    b = g[g.label_binary == "benign"]
    ben_src.update(b.src_ip.dropna().unique())
    ben_dst.update(b.dst_ip.dropna().unique())
    meta_parts.append(g[g.label_binary != "benign"][["src_ip", "incident_ids"]])
meta = pd.concat(meta_parts, ignore_index=True)
assert len(meta) == len(pool) and (meta.src_ip.fillna("").values == pool.src_ip.fillna("").values).all(), "row order mismatch"
pool["incident_ids"] = meta.incident_ids.values
pool.to_parquet(INTERIM / "pool_raw.parquet", index=False)   # now includes incident_ids (split-only!)

pool["y"] = (pool.label_binary == "malicious").astype(int)
pool["hour"] = pd.to_datetime(pool.timestamp, unit="s").dt.floor("h")
pool["stream_grp"] = np.where(pool.stream_name.isin(["cisco_asa", "aws_vpc_flow_log"]), pool.stream_name, "other")

print("\nA) do pool IPs appear in benign rows?")
print("pool src_ip in benign src_ip:", round(pool.src_ip.isin(ben_src).mean(), 3),
      "| pool dst_ip in benign dst_ip:", round(pool.dst_ip.isin(ben_dst).mean(), 3))

print("\nB) malicious rows, distinct incident_ids strings, biggest incident's share, per day")
m = pool[pool.y == 1]
print(m.groupby("day").agg(rows=("y", "size"), incident_keys=("incident_ids", "nunique"),
      top_share=("incident_ids", lambda s: round(s.value_counts(normalize=True).iloc[0], 3))))
print("07-29 malicious rows by hour:", m[m.day == "07-29"].hour.dt.strftime("%H").value_counts().sort_index().to_dict())

print("\nC) cisco_asa: interface pair + access-group")
asa = pool[pool.stream_name == "cisco_asa"].copy()
x = asa.message_sanitized.str.extract(r'src ([\w-]+):[^ ]+ dst ([\w-]+):[^ ]+ by access-group "([^"]+)"')
asa["path"] = x[0].fillna("?") + ">" + x[1].fillna("?") + " " + x[2].fillna("?")
t = asa.groupby("path").y.agg(rows="size", malicious="sum")
t["mal_share"] = (t.malicious / t.rows).round(4)
print(t.sort_values("rows", ascending=False).head(8))
print(asa.assign(port53=asa.dst_port.eq("53")).groupby("port53").y.agg(rows="size", mal_share="mean").round(4))

print("\nD) exploratory behavior signal (same src_ip and hour, pool rows only)")
gb = pool.groupby(["src_ip", "hour"])
pool["n_evt"] = gb.y.transform("size")
pool["n_dst"] = gb.dst_ip.transform("nunique")
pool["n_port"] = gb.dst_port.transform("nunique")
k = ["n_evt", "n_dst", "n_port"]
print("medians:"); print(pool.groupby(["stream_grp", "label_binary"])[k].median())
print("means:");   print(pool.groupby(["stream_grp", "label_binary"])[k].mean().round(2))

##### POOL CHECK 3 #####

A) do pool IPs appear in benign rows?
pool src_ip in benign src_ip: 0.041 | pool dst_ip in benign dst_ip: 0.035

B) malicious rows, distinct incident_ids strings, biggest incident's share, per day
       rows  incident_keys  top_share
day                                  
07-26  3260            892      0.094
07-27   916            637      0.026
07-29  3515             64      0.028
07-30    37              6      0.757
07-29 malicious rows by hour: {'16': 354, '17': 68, '18': 10, '19': 7, '20': 2915, '21': 161}

C) cisco_asa: interface pair + access-group
                                  rows  malicious  mal_share
path                                                        
dmz-2>outside dmz-2_access_in_1  68315        352     0.0052
dmz-2>dmz-1 dmz-2_access_in_1    17665         19     0.0011
outside>dmz-1 outside_access_in   7191       3110     0.4325
?>? ?                             1809         34     0.0188
dmz-1>outside dmz_access_in         82      

In [4]:
print("##### POOL CHECK 3 #####")
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

ROOT = Path(r"C:\Users\Aliya\microsoft_hackathon")
SIG = ROOT / "data" / "raw" / "signals" / "signals.parquet"
INTERIM = ROOT / "data" / "interim"
pool = pd.read_parquet(INTERIM / "pool_raw.parquet")

# 1) Re-read in the same order to attach incident_ids; collect benign IPs on the way
pf = pq.ParquetFile(SIG)
meta_parts, ben_src, ben_dst = [], set(), set()
for i in range(pf.metadata.num_row_groups):
    g = pf.read_row_group(i, columns=["label_binary", "src_ip", "dst_ip", "incident_ids"]).to_pandas()
    b = g[g.label_binary == "benign"]
    ben_src.update(b.src_ip.dropna().unique())
    ben_dst.update(b.dst_ip.dropna().unique())
    meta_parts.append(g[g.label_binary != "benign"][["src_ip", "incident_ids"]])
meta = pd.concat(meta_parts, ignore_index=True)
assert len(meta) == len(pool) and (meta.src_ip.fillna("").values == pool.src_ip.fillna("").values).all(), "row order mismatch"
pool["incident_ids"] = meta.incident_ids.values
pool.to_parquet(INTERIM / "pool_raw.parquet", index=False)   # now includes incident_ids (split-only!)

pool["y"] = (pool.label_binary == "malicious").astype(int)
pool["hour"] = pd.to_datetime(pool.timestamp, unit="s").dt.floor("h")
pool["stream_grp"] = np.where(pool.stream_name.isin(["cisco_asa", "aws_vpc_flow_log"]), pool.stream_name, "other")

print("\nA) do pool IPs appear in benign rows?")
print("pool src_ip in benign src_ip:", round(pool.src_ip.isin(ben_src).mean(), 3),
      "| pool dst_ip in benign dst_ip:", round(pool.dst_ip.isin(ben_dst).mean(), 3))

print("\nB) malicious rows, distinct incident_ids strings, biggest incident's share, per day")
m = pool[pool.y == 1]
print(m.groupby("day").agg(rows=("y", "size"), incident_keys=("incident_ids", "nunique"),
      top_share=("incident_ids", lambda s: round(s.value_counts(normalize=True).iloc[0], 3))))
print("07-29 malicious rows by hour:", m[m.day == "07-29"].hour.dt.strftime("%H").value_counts().sort_index().to_dict())

print("\nC) cisco_asa: interface pair + access-group")
asa = pool[pool.stream_name == "cisco_asa"].copy()
x = asa.message_sanitized.str.extract(r'src ([\w-]+):[^ ]+ dst ([\w-]+):[^ ]+ by access-group "([^"]+)"')
asa["path"] = x[0].fillna("?") + ">" + x[1].fillna("?") + " " + x[2].fillna("?")
t = asa.groupby("path").y.agg(rows="size", malicious="sum")
t["mal_share"] = (t.malicious / t.rows).round(4)
print(t.sort_values("rows", ascending=False).head(8))
print(asa.assign(port53=asa.dst_port.eq("53")).groupby("port53").y.agg(rows="size", mal_share="mean").round(4))

print("\nD) exploratory behavior signal (same src_ip and hour, pool rows only)")
gb = pool.groupby(["src_ip", "hour"])
pool["n_evt"] = gb.y.transform("size")
pool["n_dst"] = gb.dst_ip.transform("nunique")
pool["n_port"] = gb.dst_port.transform("nunique")
k = ["n_evt", "n_dst", "n_port"]
print("medians:"); print(pool.groupby(["stream_grp", "label_binary"])[k].median())
print("means:");   print(pool.groupby(["stream_grp", "label_binary"])[k].mean().round(2))

##### POOL CHECK 3 #####

A) do pool IPs appear in benign rows?
pool src_ip in benign src_ip: 0.041 | pool dst_ip in benign dst_ip: 0.035

B) malicious rows, distinct incident_ids strings, biggest incident's share, per day
       rows  incident_keys  top_share
day                                  
07-26  3260            892      0.094
07-27   916            637      0.026
07-29  3515             64      0.028
07-30    37              6      0.757
07-29 malicious rows by hour: {'16': 354, '17': 68, '18': 10, '19': 7, '20': 2915, '21': 161}

C) cisco_asa: interface pair + access-group
                                  rows  malicious  mal_share
path                                                        
dmz-2>outside dmz-2_access_in_1  68315        352     0.0052
dmz-2>dmz-1 dmz-2_access_in_1    17665         19     0.0011
outside>dmz-1 outside_access_in   7191       3110     0.4325
?>? ?                             1809         34     0.0188
dmz-1>outside dmz_access_in         82      

In [5]:
print("##### SPLIT PLANNING #####")
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)

ROOT = Path(r"C:\Users\Aliya\microsoft_hackathon")
pool = pd.read_parquet(ROOT / "data" / "interim" / "pool_raw.parquet")
pool["y"] = (pool.label_binary == "malicious").astype(int)
dt = pd.to_datetime(pool.timestamp, unit="s")
pool["day"], pool["hour"] = dt.dt.strftime("%m-%d"), dt.dt.hour

def hourly(stream, days):
    d = pool[(pool.stream_name == stream) & pool.day.isin(days)]
    t = d.groupby(["day", "hour"]).y.agg(mal="sum", rows="size")
    print(f"\n=== {stream}: malicious / rows per hour (UTC) ===")
    print(t.T.to_string())

hourly("cisco_asa", ["07-26", "07-27"])
hourly("aws_vpc_flow_log", ["07-26", "07-27", "07-29", "07-30"])

print("\n=== incident-key overlap across proposed VPC cuts (exact-string match, so a lower bound) ===")
v = pool[(pool.stream_name == "aws_vpc_flow_log") & (pool.y == 1)]
tr = set(v[v.day.isin(["07-26", "07-27"])].incident_ids)
va = set(v[(v.day == "07-29") & (v.hour < 20)].incident_ids)
te = set(v[((v.day == "07-29") & (v.hour >= 20)) | (v.day == "07-30")].incident_ids)
print("keys train/val/test:", len(tr), len(va), len(te))
print("train&test:", len(tr & te), "| val&test:", len(va & te), "| train&val:", len(tr & va))

print("\n=== incident-key overlap, ASA 07-26 vs 07-27 ===")
a = pool[(pool.stream_name == "cisco_asa") & (pool.y == 1)]
k26, k27 = set(a[a.day == "07-26"].incident_ids), set(a[a.day == "07-27"].incident_ids)
print("keys 07-26 / 07-27:", len(k26), len(k27), "| overlap:", len(k26 & k27))

##### SPLIT PLANNING #####

=== cisco_asa: malicious / rows per hour (UTC) ===
day  07-26                                            07-27                                  
hour    11    12    13  14    17    18    19    20 21    0     1     2     3     4     5   6 
mal   1428   571   117   0   103   174   183    75  0     0   142   205   148   180   199   0
rows  8915  8336  7267  29  6608  8522  8626  1997  1   707  8731  8922  8688  8940  8760  13

=== aws_vpc_flow_log: malicious / rows per hour (UTC) ===
day  07-26    07-27                               07-29                             07-30                       
hour    11 13    1   4   5  6   7  8   10 16   20    14    16   17   18    20    21    4   5   6   10  11  12 16
mal    593  0     0   0   0  0   0  0   0  0   18     0   354   68   10  2915   161    37   0   0   0   0   0  0
rows  2975  3    24  13  80  7  31  5  12  3  795     5  1264  334  300  9920  2395   756  11  10  23  30  11  4

=== incident-key overlap across p